# Mi proyecto · De la teoría a la práctica (Sesiones 1 a 3)
**Plantilla espejo del caso guiado Olist** · Módulo: Preparación, Recopilación y Análisis Exploratorio de Datos · MIADAS – UMSA

**Grupo:** _5_

**Integrantes:**

_Katherine Deina Nina Alejo_

_Boris Iván Cruz Torrico_

_Carlos Pacha Córdova_

_Gabriel Marcelo Alejo Nina_

**Conjunto de datos:** _E-Commerce Sales Analytics Dataset_ ·

**Fuente y URL:** _https://www.kaggle.com/datasets/datascikhan/e-commerce-sales-and-customer-analytics?select=order_items.csv_


## Paso 0 · Cargar los datos

In [1]:
import math
import pandas as pd

BASE = "https://raw.githubusercontent.com/KatherineNina/miadas-grupo5-proyecto/main/E-Commerce/"

pedidos = pd.read_csv(BASE + "ecommerce_sales_customer_analytics_150k.csv.gz")
clientes = pd.read_csv(BASE + "customer_master.csv")
items = pd.read_csv(BASE + "order_items.csv.gz")
productos = pd.read_csv(BASE + "product_catalog.csv")
estadisticas = pd.read_csv(BASE + "dataset_statistics.csv")

print("Pedidos:     ", pedidos.shape)
print("Clientes:    ", clientes.shape)
print("Ítems:       ", items.shape)
print("Productos:   ", productos.shape)
print("Estadísticas:", estadisticas.shape)


Pedidos:      (138116, 46)
Clientes:     (25000, 11)
Ítems:        (397569, 12)
Productos:    (1175, 9)
Estadísticas: (1, 11)


---
# S1 · Tipos de datos y escalas (Sesión 1)
### S1.1 · Tipos de dato
¿Qué columnas se leyeron con un tipo incorrecto (fechas como texto, códigos como números)?

S1.1 · Primera mirada a los pedidos

Mediante las funciones head() e info() de la biblioteca Pandas se realiza una exploración inicial del conjunto de datos E-Commerce Sales & Customer Analytics.

La función head(3) permite visualizar los tres primeros registros de la tabla de pedidos, mientras que info() proporciona información sobre el número de registros, las columnas, sus tipos de datos y los valores no nulos.

In [2]:
pedidos.head(3)

,order_id,order_date,order_time,order_status,sales_channel,customer_id,customer_name,customer_age,gender,customer_segment,...,discount_amount,tax_amount,shipping_cost,net_sales,product_cost,profit,profit_margin_percentage,customer_lifetime_value,is_repeat_customer,customer_order_count
0,ORD-301242,2023-11-06,16:37:47,Completed,Mobile App,CUST-003102,Jasmine Ryan,55,Male,Consumer,...,477.89,61.07,12.03,945.40,588.33,345.04,36.50,12459.68,True,11
1,ORD-773460,2025-12-24,01:22:36,Completed,Website,CUST-003124,Scott Chase,26,Male,Premium,...,1456.06,320.83,9.00,2018.41,2031.88,-22.47,-1.11,13032.48,True,11
2,ORD-449374,2021-07-05,14:24:21,Completed,Mobile App,CUST-012496,Marc Wheeler,69,Female,Premium,...,97.30,29.79,13.05,468.35,257.73,197.57,42.18,6159.44,True,6


In [3]:
pedidos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138116 entries, 0 to 138115
Data columns (total 46 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   order_id                  138116 non-null  object 
 1   order_date                138116 non-null  object 
 2   order_time                138116 non-null  object 
 3   order_status              138116 non-null  object 
 4   sales_channel             138116 non-null  object 
 5   customer_id               138116 non-null  object 
 6   customer_name             138116 non-null  object 
 7   customer_age              138116 non-null  int64  
 8   gender                    138116 non-null  object 
 9   customer_segment          138116 non-null  object 
 10  customer_type             138116 non-null  object 
 11  customer_city             138116 non-null  object 
 12  customer_state            138116 non-null  object 
 13  customer_country          138116 non-null  o

In [4]:
# Identificar columnas relacionadas con fechas
columnas_fecha = [
    col for col in pedidos.columns
    if "date" in col.lower() or "timestamp" in col.lower()
]

print("Columnas de fecha identificadas:")
print(columnas_fecha)

Columnas de fecha identificadas:
['order_date']


In [5]:
# Convertir las columnas identificadas a tipo fecha

for col in columnas_fecha:
    pedidos[col] = pd.to_datetime(
        pedidos[col],
        errors="coerce"
    )

# Verificar los nuevos tipos de datos
print(pedidos[columnas_fecha].dtypes)

# Verificar el período de los pedidos
print("Primer pedido:", pedidos["order_date"].min())
print("Último pedido:", pedidos["order_date"].max())

order_date    datetime64[ns]
dtype: object
Primer pedido: 2021-01-01 00:00:00
Último pedido: 2025-12-31 00:00:00


S1.2 · Corrección de tipos de datos

Al revisar los datos de clientes y pedidos, queremos comprobar si los códigos postales y los identificadores tienen el tipo de dato adecuado.

Aunque algunos de estos valores contienen números, no representan cantidades, sino que sirven para identificar clientes, pedidos o ubicaciones. Por eso, vamos a revisar cómo fueron cargados y corregirlos si es necesario.

In [6]:
# S1.2 - Revisar los tipos de datos originales

clientes_original = pd.read_csv(BASE + "customer_master.csv")
pedidos_original = pd.read_csv(
    BASE + "ecommerce_sales_customer_analytics_150k.csv.gz"
)

In [7]:
# Revisar los tipos de datos
print("Tipos originales de clientes:")
print(clientes_original[[
    "customer_id",
    "customer_postal_code"
]].dtypes)

print("\nTipos originales de pedidos:")
print(pedidos_original[[
    "order_id",
    "customer_id"
]].dtypes)


Tipos originales de clientes:
customer_id             object
customer_postal_code     int64
dtype: object

Tipos originales de pedidos:
order_id       object
customer_id    object
dtype: object


In [8]:
# Observar los primeros registros
clientes_original[[
    "customer_id",
    "customer_postal_code",
    "customer_city",
    "customer_state"
]].head(3)

,customer_id,customer_postal_code,customer_city,customer_state
0,CUST-000001,11253,Laurenland,Dubai
1,CUST-000002,47778,Lake Peter,Lower Saxony
2,CUST-000003,13900,Seanview,North Rhine


In [9]:
# S1.2 - Corregir los tipos de datos

# Convertir códigos postales a texto
clientes["customer_postal_code"] = (
    clientes["customer_postal_code"].astype("string")
)

# Convertir identificadores a texto
clientes["customer_id"] = clientes["customer_id"].astype("string")

pedidos["order_date"] = pd.to_datetime(pedidos["order_date"], errors="coerce")

for col in ["order_id", "customer_id", "customer_postal_code"]:
    pedidos[col] = pedidos[col].astype("string")

items["order_id"]       = items["order_id"].astype("string")
items["product_id"]     = items["product_id"].astype("string")
productos["product_id"] = productos["product_id"].astype("string")

In [11]:
# S1.2 - Verificar los tipos corregidos

print("Tipos corregidos de clientes:")
print(clientes[[
    "customer_id",
    "customer_postal_code"
]].dtypes)

print("\nTipos corregidos de pedidos:")
print(pedidos[[
    "order_id",
    "customer_id"
]].dtypes)

Tipos corregidos de clientes:
customer_id             string[python]
customer_postal_code    string[python]
dtype: object

Tipos corregidos de pedidos:
order_id       string[python]
customer_id    string[python]
dtype: object


In [12]:
# Mostrar los primeros registros corregidos
clientes[[
    "customer_id",
    "customer_postal_code",
    "customer_city",
    "customer_state"
]].head(3)

,customer_id,customer_postal_code,customer_city,customer_state
0,CUST-000001,11253,Laurenland,Dubai
1,CUST-000002,47778,Lake Peter,Lower Saxony
2,CUST-000003,13900,Seanview,North Rhine


Se observa:

Al revisar los datos, podemos ver que los códigos postales y los identificadores se utilizan para reconocer registros y no para realizar cálculos.

Por eso, los manejamos como texto utilizando astype("string"). Después de realizar la conversión, comprobamos que estas columnas tienen el tipo de dato adecuado.

De esta manera, evitamos posibles problemas al trabajar con los identificadores y dejamos las tablas preparadas para continuar con el análisis.

S1.3 · Escala de cada variable

En esta parte vamos a identificar las escalas de medición de algunas variables del dataset de E-Commerce. Para ello, seleccionamos variables de las tablas de pedidos, clientes, productos e ítems y las clasificamos según su escala de medición.

| Variable | Escala (nominal / ordinal / intervalo / razón) | Cálculos válidos |
|---|---|---|
| `order_id` | Nominal | Conteos, frecuencias |
| `customer_id` | Nominal | Conteos, clientes únicos |
| `customer_postal_code` | Nominal | Frecuencias |
| `customer_segment` | Nominal | Frecuencias, moda |
| `order_status` | Nominal | Frecuencias, moda |
| `product_category` | Nominal | Frecuencias, moda |
| `order_date` | Intervalo | Diferencias entre fechas |
| `quantity` | Razón | Suma, promedio, mediana |
| `unit_price` | Razón | Promedio, mediana |
| `net_sales` | Razón | Suma, promedio, mediana |

Primero vamos a revisar la variable order_status, que nos muestra el estado de cada pedido. Como es una variable nominal, podemos contar cuántos pedidos pertenecen a cada categoría.

Para esto utilizaremos value_counts(), que nos permitirá conocer la frecuencia de cada estado.

In [13]:
# S1.3 - Frecuencias de una variable nominal

pedidos["order_status"].value_counts()

,count
order_status,
Completed,113559
Returned,9462
Cancelled,8398
Pending,6697


Al revisar los resultados, podemos ver que la mayoría de los pedidos están completados, con 113.559 registros, lo que representa aproximadamente el 82,2% del total.

También encontramos pedidos devueltos, cancelados y pendientes, pero en cantidades mucho menores.

Con esto podemos conocer cómo se distribuyen los pedidos según su estado y observar que la mayor parte corresponde a compras completadas.

**Análisis de variables de razón**

Ahora vamos a trabajar con dos variables numéricas de la tabla de ítems: unit_price, que representa el precio unitario de los productos, y net_sales, que corresponde a la venta neta de cada línea de pedido.

A diferencia de las variables nominales, estas sí nos permiten realizar cálculos como promedios, medianas y desviaciones estándar.

Utilizaremos describe() para conocer sus principales estadísticas y observar cómo se distribuyen sus valores.

In [14]:
# S1.3 - Estadísticas de variables de razón

items[["unit_price", "net_sales"]].describe().round(2)


,unit_price,net_sales
count,397569.00,397569.00
mean,245.19,483.94
std,251.04,615.43
min,6.31,5.35
25%,74.36,124.28
50%,164.97,271.10
75%,310.79,595.24
max,1482.95,8897.70


Al revisar los resultados, encontramos que el precio unitario promedio de los productos es de 245,19, mientras que la mediana es de 164,97. Esta diferencia nos muestra que existen productos con precios más altos que hacen subir el promedio.

En el caso de las ventas netas, el promedio es de 483,94 y la mediana de 271,10. Aquí también podemos ver una diferencia importante, lo que indica que algunas ventas tienen importes bastante mayores que otras.

Además, observamos que el precio unitario máximo llega a 1.482,95 y la venta neta máxima a 8.897,70. Estos valores podrían ser interesantes de revisar más adelante para identificar posibles datos atípicos.

---
# S2 · Tamaño de muestra y muestreo (Sesión 2)

En esta sección vamos a calcular cuántos pedidos de nuestro dataset necesitaríamos revisar para obtener una muestra representativa, sin tener que analizar manualmente todos los registros.

Para ello, utilizaremos la fórmula de Cochran con corrección por población finita, considerando un nivel de confianza del 95% y un margen de error del 5%.

Después compararemos una muestra aleatoria simple con una muestra estratificada, para observar cuál representa mejor los diferentes estados de los pedidos.

**S2.1 · ¿Cuántos pedidos necesitamos revisar?**

Supongamos que queremos revisar manualmente una parte de los pedidos para analizar su estado y conocer cuántos fueron completados, cancelados, devueltos o siguen pendientes.

Nuestro dataset contiene 138.116 pedidos. Como revisar todos sería demasiado trabajo, calcularemos una muestra utilizando un nivel de confianza del 95%, un margen de error del 5% y una proporción esperada de 0,5.

In [15]:
# S2.1 - Calcular el tamaño de muestra

def tamano_muestra(N, e=0.05, z=1.96, p=0.5):
    n0 = z**2 * p * (1 - p) / e**2
    return math.ceil(n0 / (1 + (n0 - 1) / N))

N = len(pedidos)
n = tamano_muestra(N)

print("Población:", N)
print("Tamaño de muestra:", n)


Población: 138116
Tamaño de muestra: 384


Al aplicar la fórmula, encontramos que de los 138.116 pedidos que contiene nuestro dataset necesitaríamos seleccionar una muestra de 384 pedidos.

Esto significa que no sería necesario revisar todos los registros para estimar proporciones generales, siempre que la muestra se seleccione correctamente. Para el cálculo utilizamos un nivel de confianza del 95% y un margen de error del 5%.

**S2.2 · Muestreo aleatorio y estratificado**

Ahora que conocemos el tamaño de muestra, vamos a comparar dos formas de seleccionar los 384 pedidos.

Primero utilizaremos el muestreo aleatorio simple, donde todos los pedidos tienen la misma posibilidad de ser seleccionados.

Después aplicaremos el muestreo estratificado utilizando la variable order_status, que nos permitirá separar los pedidos según su estado y seleccionar una cantidad proporcional de cada grupo.

Finalmente, compararemos los porcentajes obtenidos con los de la población original.

In [16]:
# S2.2 - Muestreo aleatorio y estratificado

variable_estrato = "order_status"

# Muestreo aleatorio simple
aleatoria = pedidos.sample(n=n, random_state=42)

# Muestreo estratificado proporcional
proporciones = pedidos[variable_estrato].value_counts(
    normalize=True
)

In [17]:
# Calcular cantidad por estrato
cuotas = proporciones * n
cantidades = cuotas.astype(int)

# Distribuir los registros restantes
restantes = n - cantidades.sum()
orden = (cuotas - cantidades).sort_values(
    ascending=False
).index

cantidades.loc[orden[:restantes]] += 1

# Seleccionar los pedidos de cada estrato
estratificada = pd.concat([
    grupo.sample(
        n=int(cantidades.loc[estado]),
        random_state=42
    )
    for estado, grupo in pedidos.groupby(variable_estrato)
])

print("Muestra aleatoria:", len(aleatoria))
print("Muestra estratificada:", len(estratificada))

Muestra aleatoria: 384
Muestra estratificada: 384


**S2.3 · Comparación de las muestras**

Una vez seleccionadas las dos muestras, vamos a comparar la distribución de los estados de los pedidos.

Para esto calcularemos el porcentaje que representa cada estado en la población completa, en la muestra aleatoria y en la muestra estratificada.

De esta manera podremos observar qué método mantiene una distribución más parecida a la de nuestros datos originales.

In [18]:
# S2.3 - Comparar las distribuciones

comparacion = pd.DataFrame({
    "Población": pedidos[variable_estrato].value_counts(
        normalize=True
    ),
    "Aleatoria": aleatoria[variable_estrato].value_counts(
        normalize=True
    ),
    "Estratificada": estratificada[variable_estrato].value_counts(
        normalize=True
    )
}).fillna(0).mul(100).round(2)

comparacion

,Población,Aleatoria,Estratificada
order_status,,,
Cancelled,6.08,7.03,5.99
Completed,82.22,82.55,82.29
Pending,4.85,3.65,4.95
Returned,6.85,6.77,6.77


Al comparar los resultados, podemos ver que ambas muestras tienen porcentajes parecidos a los de la población original, aunque existen algunas diferencias: En la muestra aleatoria, por ejemplo, los pedidos pendientes representan el 3,65%, mientras que en la población total son el 4,85%. En cambio, con el muestreo estratificado obtenemos un 4,95%, que se acerca más al valor original.

También observamos que los pedidos completados representan aproximadamente el 82% en los tres casos.

Con estos resultados podemos ver que el muestreo estratificado mantiene mejor las proporciones de los diferentes estados de los pedidos, lo que puede ser útil cuando queremos asegurarnos de que todos los grupos estén representados de manera proporcional.

---
# S3 · Fuentes e integración (Sesión 3)

En esta sección vamos a trabajar con las diferentes tablas de nuestro dataset de E-Commerce y revisar cómo podemos relacionarlas entre sí.

Hasta ahora hemos analizado algunas variables por separado, pero también podemos unir la información de pedidos, clientes, productos e ítems para obtener resultados más completos.

Para ello, utilizaremos los identificadores que tienen en común las tablas y la función merge() de Pandas, que nos permitirá combinar la información.

**S3.1 · Modelo de datos de E-Commerce**

Nuestro conjunto de datos está organizado en cinco tablas. Para realizar el análisis utilizaremos principalmente cuatro: pedidos, clientes, ítems y productos.

Estas tablas se pueden relacionar mediante los identificadores customer_id, order_id y product_id.

De esta manera, podremos conocer no solamente la información de cada pedido, sino también a qué cliente pertenece y qué productos se compraron.

```text
            ┌──────────────────────┐
            │       CLIENTES       │
            │     customer_id      │
            └──────────┬───────────┘
                       │
                  customer_id
                       │
            ┌──────────▼───────────┐
            │        PEDIDOS       │
            │       order_id       │
            │      customer_id     │
            └──────────┬───────────┘
                       │
                    order_id
                       │
            ┌──────────▼───────────┐
            │         ÍTEMS        │
            │       order_id       │
            │      product_id      │
            └──────────┬───────────┘
                       │
                   product_id
                       │
            ┌──────────▼───────────┐
            │       PRODUCTOS      │
            │      product_id      │
            └──────────────────────┘
```

**S3.2 · ¿De qué países provienen los pedidos?**

Primero vamos a relacionar la tabla de pedidos con la tabla de clientes mediante customer_id.

Esto nos permitirá identificar el país de los clientes y conocer de dónde proviene la mayor cantidad de pedidos.

Después utilizaremos value_counts() para observar los cinco países con más pedidos registrados.

In [19]:
# S3.2 - Integrar pedidos con clientes

pedidos_clientes = pedidos.merge(
    clientes[["customer_id", "customer_country"]],
    on="customer_id",
    how="left",
    suffixes=("", "_maestro"),
    validate="many_to_one"
)

# Verificar la integración
print("Pedidos originales:", len(pedidos))
print("Pedidos integrados:", len(pedidos_clientes))

# Países con más pedidos
pedidos_clientes["customer_country_maestro"].value_counts().head(5)


Pedidos originales: 138116
Pedidos integrados: 138116


,count
customer_country_maestro,
USA,82600
UK,20235
Germany,11593
Canada,7015
Australia,6794


**S3.3 · Integración de pedidos, ítems y productos**

Después de relacionar los pedidos con los clientes, vamos a incorporar la información de los productos comprados.

Para ello, utilizaremos order_id para unir los pedidos con sus ítems y product_id para relacionar cada ítem con el catálogo de productos.

Con esta integración podremos analizar las ventas por producto, categoría y características de los clientes.

In [20]:

# S3.3 - Integrar las cuatro tablas principales

integrado = (
    pedidos_clientes
    .merge(
        items,
        on="order_id",
        how="left",
        suffixes=("_pedido", "_item"),
        validate="one_to_many"
    )
    .merge(
        productos[[
            "product_id",
            "product_name",
            "product_category",
            "product_subcategory",
            "brand"
        ]],
        on="product_id",
        how="left",
        validate="many_to_one"
    )
)

In [21]:

# Comprobar el resultado
print("Dimensión de la tabla integrada:", integrado.shape)

print("Pedidos únicos:", integrado["order_id"].nunique())
print("Clientes únicos:", integrado["customer_id"].nunique())
print("Productos únicos:", integrado["product_id"].nunique())

# Mostrar los primeros registros
integrado[[
    "order_id",
    "customer_id",
    "product_id",
    "product_name",
    "product_category",
    "quantity_item",
    "net_sales_item"
]].head(5)

Dimensión de la tabla integrada: (397569, 62)
Pedidos únicos: 138116
Clientes únicos: 24911
Productos únicos: 1175


,order_id,customer_id,product_id,product_name,product_category,quantity_item,net_sales_item
0,ORD-301242,CUST-003102,PROD-000738,Garmin Total client-driven task-force Medical ...,Health & Wellness,2,351.88
1,ORD-301242,CUST-003102,PROD-000181,Tommy Hilfiger Open-architected full-range foc...,Fashion,3,593.52
2,ORD-773460,CUST-003124,PROD-000973,Swarovski Re-engineered discrete monitoring Ea...,Jewelry,4,1919.32
3,ORD-773460,CUST-003124,PROD-000268,NARS Enterprise-wide actuating firmware Fragra...,Beauty & Personal Care,4,99.09
4,ORD-449374,CUST-012496,PROD-001040,Mars Multi-tiered homogeneous artificial intel...,Grocery,3,19.42


**S3.4 · ¿Cuántos pedidos llegaron tarde?**

Ahora vamos a revisar si los pedidos completados fueron entregados dentro del tiempo estimado.

Para ello, compararemos los días que tardó cada entrega con los días previstos. Consideraremos que un pedido llegó tarde cuando el tiempo real de entrega sea mayor al estimado.

Con esto podremos conocer cuántos pedidos tuvieron retrasos y qué porcentaje representan.

In [22]:
# S3.4 - Analizar pedidos entregados tarde

entregados = pedidos[
    (pedidos["order_status"] == "Completed") &
    pedidos["delivery_days"].notna() &
    pedidos["estimated_delivery_days"].notna()
].copy()

# Identificar entregas tardías
entregados["tarde"] = (
    entregados["delivery_days"] >
    entregados["estimated_delivery_days"]
)

print("Pedidos completados:", len(entregados))
print("Llegaron tarde:", entregados["tarde"].sum())

print(
    "Porcentaje tarde:",
    round(entregados["tarde"].mean() * 100, 2), "%"
)

Pedidos completados: 113559
Llegaron tarde: 16886
Porcentaje tarde: 14.87 %


---
# Cierre · Informe de recolección inicial (CRISP-DM, fase 2)


| Elemento | Mi conjunto de datos |
|---|---|
| **Fuente** | E-Commerce Sales & Customer Analytics, disponible en Kaggle y almacenado en nuestro repositorio de GitHub. |
| **Tipo de fuente (interna / externa · primaria / secundaria)** | Externa y secundaria. Dataset sintético utilizado para análisis académico. |
| **Formato y acceso (CSV, Excel, API... URL)** | Cinco archivos CSV descargados de Kaggle y cargados desde GitHub con `pd.read_csv()`. Dos archivos están comprimidos en formato `.csv.gz`. |
| **Tablas y número de registros** | Pedidos: 138.116; clientes: 25.000; ítems: 397.569; productos: 1.175; estadísticas: 1. |
| **Período que cubre** | Del 1 de enero de 2021 al 31 de diciembre de 2025. |
| **Problemas encontrados** | Algunas fechas se cargaron como texto y fue necesario convertirlas. También revisamos los identificadores y códigos postales para tratarlos como texto. La integración requiere controlar los registros repetidos por pedido. |
| **Primer hallazgo o pregunta de negocio** | De los 113.559 pedidos completados, 16.886 llegaron tarde (14,87%). Esto nos lleva a preguntarnos qué factores están relacionados con los retrasos en las entregas. |
